# Where this actually fits

> A working classifier in eight lines, the only vocabulary you actually need, and the honest answer to why none of this worked until about ten minutes ago.

Read this chapter at `/learn/01-where-this-fits/`. Exported from `src/content/chapters/01-where-this-fits.mdx` — edit there, not here.


Before any explaining, here is a machine learning model. Press **Run**.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

digits = load_digits()                       # 1797 hand-written digits, 8x8 pixels
X_train, X_valid, y_train, y_valid = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=0)

model = LogisticRegression(max_iter=5000).fit(X_train, y_train)
print(f"accuracy on digits it has never seen: {model.score(X_valid, y_valid):.1%}")

The first time it runs it takes a few seconds, because your browser is quietly
downloading a whole Python. After that it's instant. Go on — it's already done.

So. That thing just read handwriting. Not handwriting it had seen before —
handwriting it had never laid eyes on — and it got about nineteen out of twenty
right.

Now here is the part I'd love you to sit with for a second, because it is the
strangest thing on this page and it's easy to read straight past.

**Nobody told it what a 7 looks like.**

Nowhere in those eight lines did anyone describe a stroke, or handle the case
where the crossbar is missing, or write a rule about the little hook some people
put on a 9. There isn't a single `if` statement about digits anywhere. There
isn't a single *fact about digits* anywhere. And yet.

You didn't write the rules. You handed over examples, and something went and
found the rules for you. That inversion is the entire subject. Everything else —
all sixteen chapters of it — is mechanism.

## The inversion

Here's a shape you know extremely well:

<div class="table-scroll">

| | You provide | The computer gives back |
|---|---|---|
| **Ordinary program** | rules + input | output |
| **Machine learning** | input + output | **rules** |

</div>

Read the second row again. You hand over the questions *and* the answers, and
what comes back is the method.

That's a genuinely odd trade, and the reason it's worth making is something you
already know in your bones. You've written the top row for years, and you know
exactly which problems it's bad at: the ones where you can't state the rule.

You can specify a JSON parser completely. Sit down, think hard, and every case is
writable. Now try to specify "is this photo a cat." Not *recognise* one — you can
do that in 40 milliseconds without trying. **Specify** it. Write the conditions.

You can't. Not because you don't know a cat when you see one, but because the
knowing isn't in a form your fingers can type. It's in there, and it won't come
out.

Machine learning is, mostly, a set of tools for getting knowledge out of that
locked room by showing examples through the window.

The closest thing you already write is the gap between a function and a
*constraint*. You know the signature you want:

`fn is_seven(pixels: &[u8]) -> bool`

You have no idea what goes in the body, and no amount of staring will produce
it. What you *can* produce is a pile of `(input, expected)` pairs and a way to
score how badly a candidate body is doing. Machine learning fills in the body by
searching, guided by that score.

## The four words, once, and then we'll stop

Four words get thrown around in public as if they mean the same thing. They
don't. Five minutes here saves you a lot of confusion later, and then we mostly
won't need them again.

**Artificial intelligence** is the whole field, and it's old — the name was
coined at a workshop in 1956. It covers chess engines built entirely from
hand-written search, expert systems built from hand-written rules, and everything
below. It's an *aspiration*, not a technique. Almost nobody describes their own
work as "AI" when talking to another practitioner.

**Machine learning** is the part of AI where the rules get fitted to data
instead of written by hand. Logistic regression is machine learning. So is a
decision tree. So was the spam filter that shipped in 2002. Most of it involves
no neural networks at all.

**Deep learning** is the part of machine learning that uses neural networks with
many layers. "Deep" literally means "has a lot of layers stacked up." That's the
whole etymology. It's the part that got startlingly good after about 2012, and
it's where nearly all the recent noise lives.

**A model** is the fitted thing itself — the function with its numbers filled in.
`model` in the code above is a model. A file of weights is a model. When someone
says "we deployed the model," they shipped a function.

An unfortunate consequence of that nesting: a linear regression from 1805 is,
technically and correctly, artificial intelligence. Which means "AI-powered" is a
claim entirely compatible with a spreadsheet formula.

Being able to ask *"which layer of that nest do you mean?"* — politely, in a
meeting — turns out to be a genuinely useful professional skill.

Three more words and we can get back to the interesting part:

- **Training** — finding the numbers. Expensive, done once, offline.
- **Inference** — running the finished model on something new. Cheap, done constantly.
- **Parameters** (or *weights*) — the numbers that got found. The model above found
  650 of them. The big language models have hundreds of billions.

## Let's take it apart

I don't want you to take my word for any of this, so let's open the model up. It
will not take long, because there is almost nothing inside.

In [ ]:
w = model.coef_          # the learned weights
b = model.intercept_     # the learned offsets
print("weights:", w.shape, " offsets:", b.shape)
print("total learned numbers:", w.size + b.size)

Ten rows of 64 weights, plus ten offsets. One row per digit, one weight per
pixel. That's it. That is the entire model — 650 floating-point numbers.
Everything it learned from 1257 examples of human handwriting is sitting in
those 650 numbers and nowhere else.

To classify an image it scores each of the ten digits — multiply every pixel by
that digit's weight, add them up — and picks the winner. Let's do it by hand,
with no scikit-learn anywhere, just so you can see there's nothing up my sleeve.

In [ ]:
import numpy as np

image = X_valid[0]                    # one 8x8 digit, flattened to 64 numbers
scores = image @ w.T + b              # ten scores, one per digit
print("scores :", np.round(scores, 1))
print("predicted:", scores.argmax(), " actual:", y_valid[0])

One matrix multiply, one addition, one
argmax. That is *inference* — the whole of it.

Which means the expensive part was never running the model. Running it costs a
handful of multiplications. The expensive part was finding those 650 numbers, and
that asymmetry — costly once, cheap forever after — is why any of this is
economically interesting at all.

Here's the nice way to see it.

Each row of `w` is a **template** — a 64-number picture of what that digit tends
to look like. The  between an image and a template comes
out large when the bright pixels of the image line up with the large weights of
the template.

So "which digit is this?" quietly becomes "which template does this image agree
with most?", and agreement is measured by a dot product. That's all a linear
classifier ever is: a shelf of templates and a way to measure agreement.

And you don't have to believe me, because the templates are pictures and we can
just look at them:

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 5, figsize=(7, 3))
for digit, ax in enumerate(axes.flat):
    ax.imshow(w[digit].reshape(8, 8), cmap="RdBu_r")
    ax.set_title(str(digit), fontsize=9); ax.axis("off")
plt.tight_layout()

Red means "a bright pixel here argues *for* this digit." Blue means it argues
against.

Look at the 0: it's a ring, and the middle is blue. The model worked out that a
zero is a shape with a *hole*, and that ink in the middle is evidence against.
Look at the 1: a vertical stripe. Nobody wrote either of those down. They fell
out of 1257 examples and some arithmetic.

And here — right here — is the limitation, visible in the same picture. One
template per class cannot say "a 7 with a crossbar *or* a 7 without one." It has
to average them, and the average of two different-looking 7s is a blurry 7 that
matches neither very well. You can see it in the plot if you look: the harder
digits are muddier.

Fixing exactly that is what layers are for, and that's
[Chapter 8](/learn/08-neural-networks/). Keep this picture in mind until then.

**"I ran it and got a slightly different number."** Good — that means you're
paying attention. Small differences come from library versions and the browser's
maths. Anything in the 94–96% range is the same result. If you get something
wildly different, something is genuinely wrong and I'd want to know.

**"`@` is a matrix multiply? Since when?"** Since Python 3.5. It's a real
operator that exists mostly for this. Hover the underlined
matrix multiply above for the two-line version.

**"I don't see how multiplying pixels by numbers 'recognises' anything."** You're
not missing something — that reaction is correct and it's the right thing to be
suspicious of. Open the fold above and *look at the templates*. That's the
moment it usually clicks: the numbers are a picture, and the multiplication is
asking "how much does this look like that?"

**"Where did the 650 numbers come from?"** Deliberately not answered yet. That's
chapters 4 and 5, and it's the best part. For today all I want is for you to
believe they're *just numbers*, and that a model is a function someone found
rather than a mind someone summoned.

## So why didn't this work until about ten minutes ago?

This is the question I'd most like you to leave today with, because the answer
isn't the one most people assume, and it tells you something about where the
field goes next.

Here's the surprise: **the ideas are old.**

The perceptron — a single artificial neuron, trained by exactly the method you'll
build in chapter 5 — was built as *physical hardware* in 1958. Backpropagation,
which makes deep networks trainable and which we'll write by hand in
[Chapter 9](/learn/09-backpropagation/), was popularised in 1986. Convolutional
networks were in production reading cheques for the US Postal Service in the
early 1990s. LSTMs, which handled sequences for the next two decades, are from
1997.

So "we finally thought of it" is not the answer. The thinking was done. Three
other things had to show up.

**Data.** A model fits itself to examples, so it needs examples, and for most of
this history nobody had any. The dataset that broke the field open was ImageNet:
14 million labelled photographs, assembled between 2007 and 2009, in large part
by paying people on Mechanical Turk to label images one at a time. It took the
internet to make collection cheap and crowdsourcing to make labelling cheap.
Before that, nobody on Earth had a million labelled anything.

**Compute — and specifically the right *shape* of compute.** You saw above that
inference is a matrix multiply. Training is an enormous number of them. And it
turns out somebody had already built a machine for doing vast numbers of parallel
multiply-accumulates, for reasons entirely to do with drawing triangles quickly
in video games. When people started running neural networks on graphics cards
around 2009, the same experiment got roughly fifty times faster overnight.

That number matters more than it looks. An idea that takes six months to test is
not an idea you can *iterate* on. An idea that takes three days is. Fifty times
faster didn't just speed up the work; it changed what kind of work was possible.

**A handful of deeply unglamorous tricks.** Better weight initialisation. ReLU
instead of sigmoid, so gradients survive many layers
([why](/appendix/math/#relu)). Dropout. Batch normalisation. Adam. Individually
each one is a paragraph of arithmetic that would embarrass nobody. Collectively
they moved deep networks from "trainable in principle" to "trainable on a
Tuesday."

You can date the moment all three met: **September 2012**, when AlexNet won the
ImageNet competition with a 15.3% error rate against the runner-up's 26.2%. In a
mature benchmark, where everyone has been grinding out fractions of a percent for
years, a ten-point gap isn't a win. It's the sound of a field changing direction.

There's a pattern hiding in that story, and once you see it you'll see it
everywhere.

Almost every "breakthrough" in this field is an **old idea meeting newly
sufficient scale**. Transformers (2017) made attention — an idea from 2014 —
cheap enough to scale up. Diffusion models are 2015 mathematics that turned
practical around 2021. Neural networks themselves are a 1950s idea that needed
sixty years of hardware.

So when you read that something in this field is impossible, it's worth quietly
asking: *impossible, or merely currently expensive?* Those two have very
different futures, and telling them apart is most of what it means to have good
taste here.

## When the answer is not machine learning

I want to put this early rather than late, because it's where judgement lives and
almost nobody teaches it.

**When you can just write the rule.** Tax calculation is defined in legislation.
Don't learn it from examples. You'll achieve 99.4% accuracy on a problem where
100% was sitting right there for free, and you won't be able to explain the 0.6%
to anyone, least of all a regulator.

**When being wrong is unacceptable *and* unexplainable.** These models are
statistical. They are sometimes wrong and they often can't tell you why. If a
wrong answer means a wrong medication, the model is at best an input to a human
decision, and should be built and described that way.

**When you have no data.** Two hundred examples of a rare event is not a training
set, it's an anecdote. Sometimes the genuinely correct project is spending six
months building the pipeline that collects the data, and shipping the model next
year. That's a much less fun sentence to say in a planning meeting, and it is
often the right one.

**When a lookup table would do.** A startling amount of shipped "AI" is a
`GROUP BY` with better marketing. Try the boring thing first and *measure it*.
That's your baseline. If your clever model can't beat it, you found that out in
an afternoon instead of a quarter.

The expensive failure in industry is almost never choosing the wrong
architecture. It's spending four months building a model for a problem where the
data could never have supported an answer in the first place.

The skill that prevents it is framing, and it's the whole of
[Chapter 3](/learn/03-the-shape-of-problems/).

## The map

Everything so far is one branch of something bigger. Here's the whole thing.

You will not understand most of this yet. That's not a failure, it's the design —
I'd rather you have somewhere to *put* things as they arrive than meet each one
in a vacuum. Come back after every part and watch it fill in.

Three questions hang off machine learning, and running them together is the
single biggest source of early confusion:

*What feedback do you get?* (supervised, unsupervised, self-supervised,
reinforcement.) *What shape is the learned function?* (linear model, tree, neural
network.) *What makes the fitting work?* (losses, gradients, validation.)

Every project answers all three. They're independent choices. A tree can be
supervised or not; a neural network can be trained on labels or on the data
itself. Keeping the three axes apart in your head will make papers about half as
confusing, starting immediately.

Pick something you've actually built — a real feature from real work. Ask
yourself which of these it is:

1. A rule you could write down, and did.
2. A rule you could write down, but it's a horrible tangle of special cases that
   keeps growing.
3. A rule you genuinely could not write down, so the feature doesn't exist.

Category 2 is the interesting one, and it's more common than people expect.
Think about validation logic, ranking, routing, anything with a `# TODO: this
heuristic is a mess` above it.

There's no marking scheme here — the point is the habit.

Category 2 and category 3 are where machine learning earns its keep, and
category 2 is the one people miss. A tangle of special cases that keeps growing
is a rule being *fitted to data by hand, slowly, by you*. That's the same job,
done worse.

Category 1 is where people most often waste a quarter. If you can write the rule
and it's stable, write the rule.

Hold onto whichever example you picked. In chapter 3 we'll turn it into a
problem with a shape, and by chapter 16 you'll know whether it was worth doing.

## Where you're going

By the end of the fortnight you'll have built, by hand, out of nothing but
NumPy arrays: a linear model, a gradient descent
optimiser, a neural network, and the backpropagation pass that trains it.

Then you'll do the whole lot again in PyTorch in a tenth of the code — and,
because you built it once yourself, you'll know exactly what the library is doing
on your behalf. That's not a purity thing. It's the only durable way to debug
something.

You'll also be able to open a paper, read the first page, and know which boxes on
that map it lives in. That skill, more than any particular architecture, is what
makes the rest of this field learnable without anybody's help.

Tomorrow: Python, at the speed of somebody who already knows how to program.